# Test Exp13 vs Exp23 โดย Exp23 soft ensemble, 3 seeds

## 1. Import

In [ ]:
!pip install mne -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import gc
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from scipy.signal import resample_poly
import mne
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,precision_recall_fscore_support)

## 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
OLD_TEST_CSV = PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_test_ensemble" / "test_predictions.csv"
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_vs_Exp23_test"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EXP13_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed44",
}
EXP23_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed44",
}
SEEDS = [42, 43, 44]
CHECK_VAL = True

FS_RAW, FS = 256, 100
EPOCH_LEN_RAW, EPOCH_LEN = 30 * FS_RAW, 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN
ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ALL_CHANNELS
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS, N_CHANNELS = len(ALL_CHANNELS), len(SELECTED_CHANNELS)
STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
BATCH_SIZE = 80

# 3. เตรียมข้อมูล Test และคัดเลือก Subject ตามเงื่อนไข

In [ ]:
def load_split(split):
    files = sorted((DATA_DIR / split).glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"ไม่พบไฟล์ของ {split}: {DATA_DIR / split}")
    temp_dir = RESULT_DIR / f"temp_{split}"
    temp_dir.mkdir(parents=True, exist_ok=True)
    rows, temp_files = [], []

    for path in files:
        subject = path.stem
        subject_X, subject_rows = [], []
        with np.load(path, allow_pickle=True) as data:
            for g, stage in enumerate(data["stage"]):
                stage = str(stage)
                if stage not in STAGES or int(data["n_valid"][g]) != SEQ_LEN:
                    continue
                y = np.asarray(data["y"][g]).astype(str).ravel()
                epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
                x = np.asarray(data["X"][g], dtype=np.float32)
                if (len(y) != SEQ_LEN or not np.all(y == stage)
                    or len(epochs) != SEQ_LEN
                    or not np.all(np.diff(epochs) == 1)
                    or x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS)
                    or not np.isfinite(x).all()):
                    continue
                x = resample_poly(x[:, :, CHANNEL_IDX].reshape(-1, N_CHANNELS),
                                  FS, FS_RAW, axis=0).astype(np.float32)
                if x.shape != (SEQ_SAMPLES, N_CHANNELS):
                    continue
                subject_X.append(x)
                subject_rows.append({"subject": subject, "stage": stage,
                                     "label": STAGES.index(stage),
                                     "epoch_start": int(epochs[0]),
                                     "epoch_end": int(epochs[-1])})
        if {r["stage"] for r in subject_rows} == set(STAGES):
            temp_path = temp_dir / f"{subject}.npy"
            np.save(temp_path, np.stack(subject_X).astype(np.float32))
            temp_files.append((temp_path, len(subject_X)))
            rows.extend(subject_rows)
        del subject_X, subject_rows

    if not rows:
        raise ValueError(f"{split}: ไม่พบ Subject ที่มีครบ 5 ระยะ")
    total = sum(n for _, n in temp_files)
    X_path = RESULT_DIR / f"X_{split}.npy"
    X = np.lib.format.open_memmap(X_path, mode="w+", dtype=np.float32,
                                  shape=(total, SEQ_SAMPLES, N_CHANNELS))
    pos = 0
    for temp_path, n in temp_files:
        arr = np.load(temp_path, mmap_mode="r")
        X[pos:pos + n] = arr
        pos += n
        del arr
    X.flush()
    del X
    for temp_path, _ in temp_files:
        temp_path.unlink()
    temp_dir.rmdir()
    return np.load(X_path, mmap_mode="r"), pd.DataFrame(rows)

X_test, meta_test = load_split("test")
y_test = meta_test["label"].to_numpy(np.int32)
print("TEST:", meta_test["subject"].nunique(), "subjects,", len(meta_test),
      "sequences, X shape =", X_test.shape)
print(meta_test["stage"].value_counts().reindex(STAGES, fill_value=0))

def subject_names(split):
    return set(p.stem for p in (DATA_DIR / split).glob("*.npz"))
for other in ["train", "val"]:
    assert set(meta_test["subject"]).isdisjoint(subject_names(other)), f"test ซ้ำกับ {other}!"

if CHECK_VAL:
    X_val, meta_val = load_split("val")
    assert len(meta_val) == 2878

# 4. ทำนาย

In [ ]:
def predict(model, X, chunk=1000):
    return np.concatenate([model.predict(X[i:i + chunk], batch_size=BATCH_SIZE, verbose=0)
                           for i in range(0, len(X), chunk)])

def pred_table(meta, prob):
    t = meta[["subject", "epoch_start", "epoch_end"]].copy()
    t["actual"] = [STAGES[i] for i in meta["label"]]
    t["predicted"] = [STAGES[i] for i in prob.argmax(1)]
    for i, s in enumerate(STAGES):
        t["prob_" + s] = prob[:, i]
    return t

probs = {}
for group, dirs in [("Exp13", EXP13_DIRS), ("Exp23", EXP23_DIRS)]:
    for seed in SEEDS:
        name = f"{group}_s{seed}"
        model = tf.keras.models.load_model(dirs[seed] / "cnn5_best_model.keras", compile=False)
        assert model.input_shape[1:] == (SEQ_SAMPLES, N_CHANNELS), model.input_shape

        if CHECK_VAL:
            pv = predict(model, X_val).argmax(1)
            old = pd.read_csv(dirs[seed] / "val_predictions.csv")["predicted"].map(STAGES.index).to_numpy()
            agree = (pv == old).mean()
            print(f"{name}: val ตรงกับไฟล์เดิม {agree:.4f}")
            assert agree > 0.995, f"{name}: โมเดลไม่ตรง"

        probs[name] = predict(model, X_test)
        pred_table(meta_test, probs[name]).to_csv(RESULT_DIR / f"{name}_test_predictions.csv", index=False)
        del model
        tf.keras.backend.clear_session()
        gc.collect()

#5. Voting

In [ ]:
def hard_vote(names):
    P = np.stack([probs[n] for n in names])
    votes = P.argmax(2)
    counts = np.stack([(votes == k).sum(0) for k in LABELS], axis=1)
    is_tie = (counts == counts.max(1, keepdims=True)).sum(1) > 1
    pred = counts.argmax(1)
    pred[is_tie] = votes[0][is_tie]
    return pred

def soft_vote(names):
    return np.mean([probs[n] for n in names], axis=0).argmax(1)

g13 = [f"Exp13_s{s}" for s in SEEDS]
g23 = [f"Exp23_s{s}" for s in SEEDS]
preds = {n: probs[n].argmax(1) for n in probs}
preds["Exp13_hard"], preds["Exp13_soft"] = hard_vote(g13), soft_vote(g13)
preds["Exp23_hard"], preds["Exp23_soft"] = hard_vote(g23), soft_vote(g23)

# 6.ตรวจกับผล test เดิมของ Exp13

In [ ]:
old_t = pd.read_csv(OLD_TEST_CSV)
assert len(old_t) == len(y_test) and (old_t["y_true"].to_numpy() == y_test).all(), "ลำดับ/จำนวน test ไม่ตรงของเดิม"
agree_ens = (old_t["pred_ensemble"].to_numpy() == preds["Exp13_hard"]).mean()
print("Exp13 hard ensemble ตรงกับของเดิม:", round(agree_ens, 4))
for s in SEEDS:
    print(f"Exp13 seed {s} ตรงกับของเดิม:",
          round((old_t[f"pred_seed_{s}"].to_numpy() == preds[f"Exp13_s{s}"]).mean(), 4))
assert agree_ens > 0.995, "Exp13 ไม่ตรงกับผล test เดิม หยุดตรวจ"

## 7. ฟังก์ชันคำนวณ metric

In [ ]:
def calc_metrics(y_true, y_pred):
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    return ({"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p.mean(),
             "macro_recall": r.mean(), "macro_f1": f1.mean(),
             "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS)}, f1)

rows, f1_rows = {}, {}
for n, p in preds.items():
    rows[n], f1_rows[n] = calc_metrics(y_test, p)
comparison = pd.DataFrame(rows).T
f1_comparison = pd.DataFrame(f1_rows, index=STAGES).T
comparison.to_csv(RESULT_DIR / "test_comparison_overall.csv")
f1_comparison.to_csv(RESULT_DIR / "test_comparison_f1_per_class.csv")

KEY = ["Exp13_hard", "Exp13_soft", "Exp23_hard", "Exp23_soft"]
print("TEST: ผู้สมัครหลัก = Exp23_soft (เทียบกับ Exp13_hard ซึ่งเป็นตัวเลือกเดิม)")
t1 = comparison.loc[KEY, ["accuracy", "macro_f1", "cohen_kappa"]].copy()
t1["Δmacro_f1 vs Exp13_hard"] = t1["macro_f1"] - comparison.loc["Exp13_hard", "macro_f1"]
t1["Δmacro_f1 vs Exp13_soft"] = t1["macro_f1"] - comparison.loc["Exp13_soft", "macro_f1"]
display(t1.round(4))
print("F1 รายคลาส")
display(f1_comparison.loc[KEY].round(4))
print("โมเดลเดี่ยว")
display(comparison.loc[list(probs), ["accuracy", "macro_f1", "cohen_kappa"]].round(4))

cm = confusion_matrix(y_test, preds["Exp23_soft"], labels=LABELS)
print("Confusion matrix: Exp23_soft (แถว=จริง, คอลัมน์=ทาย)")
display(pd.DataFrame(cm, index=[f"จริง {s}" for s in STAGES], columns=[f"ทาย {s}" for s in STAGES]))

In [ ]:
p, r, f1, sup = precision_recall_fscore_support(y_test, preds["Exp23_soft"], labels=LABELS, zero_division=0)

report = pd.DataFrame({"precision": p, "recall": r, "f1-score": f1, "support": sup}, index=STAGES)
report.loc["macro avg"] = [p.mean(), r.mean(), f1.mean(), sup.sum()]
report.loc["weighted avg"] = [np.average(p, weights=sup), np.average(r, weights=sup), np.average(f1, weights=sup), sup.sum()]
cm_df = pd.DataFrame(cm, index=[f"จริง {s}" for s in STAGES], columns=[f"ทาย {s}" for s in STAGES])

report.to_csv(RESULT_DIR / "Exp23_soft_test_classification_report.csv")
cm_df.to_csv(RESULT_DIR / "Exp23_soft_test_confusion_matrix.csv")

display(report.round(4))
print("Saved Classification Report and Confusion Matrix")

In [ ]:
CANDIDATE = "Exp23_soft"
y_pred_vis = preds[CANDIDATE]

soft_table = meta_test.copy()
soft_table["actual"] = [STAGES[int(i)] for i in y_test]
soft_table["predicted"] = [STAGES[int(i)] for i in y_pred_vis]
soft_table["correct"] = soft_table["actual"] == soft_table["predicted"]

DATA = {"test": {"X": X_test, "y": y_test, "meta": meta_test}}
tables = {"test": soft_table}
ens_overall, _ = calc_metrics(y_test, y_pred_vis)

if "BASE_RESULT_DIR" not in globals():
    BASE_RESULT_DIR = RESULT_DIR
RESULT_DIR = BASE_RESULT_DIR / f"visual_{CANDIDATE}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
soft_table.to_csv(RESULT_DIR / "test_prediction_table.csv")
print(CANDIDATE, "| correct", int(soft_table["correct"].sum()), "/", len(soft_table))

## 8. ตารางผลทำนายราย sequence (Test)

In [ ]:
print("===== Test: Soft Voting (Exp23) Ensemble =====")
print("ทำนายถูก:", int(soft_table["correct"].sum()), "/", len(soft_table))

display(soft_table.round(3))

## 9. Hypnogram เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
print(f"Accuracy: {soft_table['correct'].mean():.4f}")
print(f"Correct: {int(soft_table['correct'].sum())}/{len(soft_table)}")

stage_y = {"W": 4, "R": 3, "N1": 2, "N2": 1, "N3": 0}

hyp_dir = RESULT_DIR / "hypnogram"
hyp_dir.mkdir(parents=True, exist_ok=True)

FULL_NIGHT_DIR = PROJECT_DIR / "dataset" / "Test_Full_night"
EPOCH_SEC = 30

FIG_W, FIG_H = 30, 10
GT_COLOR = "#1f5fff"
PRED_LINE_COLOR = "black"
LINE_W = 1.5
BLOCK_W = 8
MARKER_SIZE = 90
PRED_MARKER_SIZE = 40
DASH = (0, (5, 3))

ann_to_stage = {
    "sleep stage w": "W",
    "sleep stage n1": "N1", "sleep stage 1": "N1",
    "sleep stage n2": "N2", "sleep stage 2": "N2",
    "sleep stage n3": "N3", "sleep stage 3": "N3",
    "sleep stage r": "R",
}


def load_full_night(subject):
    ann = mne.read_annotations(str(FULL_NIGHT_DIR / f"{subject}_sleepscoring.edf"))

    items = []
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        stage = ann_to_stage.get(str(desc).strip().lower())
        if stage is None:
            continue
        items.append((int(round(onset / EPOCH_SEC)),
                      max(1, int(round(dur / EPOCH_SEC))),
                      stage_y[stage]))

    levels = np.full(max(s + n for s, n, _ in items), np.nan)
    for s, n, level in items:
        levels[s:s + n] = level
    return levels


def best_offset(levels, sub, candidates=range(-3, 4)):
    best = (0, -1.0)
    for off in candidates:
        same = total = 0
        for r in sub.itertuples():
            for e in range(int(r.epoch_start), int(r.epoch_end) + 1):
                if 0 <= e + off < len(levels):
                    total += 1
                    same += int(levels[e + off] == stage_y[r.actual])
        agree = same / total if total else 0.0
        if agree > best[1]:
            best = (off, agree)
    return best


def shift(levels, off):
    if off > 0:
        return levels[off:]
    if off < 0:
        return np.concatenate([np.full(-off, np.nan), levels])
    return levels


def fill_gaps(values):
    filled = values.copy()
    valid = np.where(~np.isnan(filled))[0]
    if len(valid) == 0:
        return filled
    filled[:valid[0]] = filled[valid[0]]
    for i in range(valid[0] + 1, len(filled)):
        if np.isnan(filled[i]):
            filled[i] = filled[i - 1]
    return filled


def build_three_lines(subject, sub):
    full_gt = load_full_night(subject)
    off, agree = best_offset(full_gt, sub)
    full_gt = shift(full_gt, off)
    n_sel_seq = len(sub)
    n_sel_ep = int((sub["epoch_end"] - sub["epoch_start"] + 1).sum())
    print(f"{subject}: epochs = {n_sel_ep} | sequence = {n_sel_seq}")

    n_ep = max(len(full_gt), int(sub["epoch_end"].max()) + 1)
    full_gt = np.concatenate([full_gt, np.full(n_ep - len(full_gt), np.nan)])

    sel_gt = np.full(n_ep, np.nan)
    pred = np.full(n_ep, np.nan)
    ok = np.full(n_ep, -1, dtype=int)

    for r in sub.itertuples():
        sl = slice(int(r.epoch_start), int(r.epoch_end) + 1)
        sel_gt[sl] = stage_y[r.actual]
        pred[sl] = stage_y[r.predicted]
        ok[sl] = int(r.correct)
    return full_gt, sel_gt, pred, ok



for subject in soft_table["subject"].unique():
    sub = soft_table[
        soft_table["subject"] == subject
    ].sort_values("epoch_start").copy()

    full_gt, sel_gt, pred, ok = build_three_lines(subject, sub)
    n_ep = len(full_gt)

    x_start = np.arange(n_ep) * EPOCH_SEC / 3600
    x_end = (np.arange(n_ep) + 1) * EPOCH_SEC / 3600
    x_step = np.append(x_start, x_end[-1])
    full_step = np.append(full_gt, full_gt[-1])

    cx = (
        sub["epoch_start"].to_numpy()
        + sub["epoch_end"].to_numpy() + 1
    ) / 2 * EPOCH_SEC / 3600

    sx0 = sub["epoch_start"].to_numpy() * EPOCH_SEC / 3600
    sx1 = (sub["epoch_end"].to_numpy() + 1) * EPOCH_SEC / 3600

    gy = sub["actual"].map(stage_y).to_numpy()
    py = sub["predicted"].map(stage_y).to_numpy()
    correct = sub["correct"].to_numpy()
    ok_color = np.where(correct, "green", "red")

    fig, axes = plt.subplots(3, 1, figsize=(FIG_W, FIG_H), sharex=True)

    axes[0].step(
        x_step, full_step, where="post",
        color="black", linewidth=LINE_W
    )

    axes[1].step(
        x_step, full_step, where="post",
        color="#9e9e9e", linewidth=LINE_W, zorder=1
    )
    axes[1].hlines(
        gy, sx0, sx1, color=GT_COLOR,
        linewidth=BLOCK_W, zorder=2
    )
    axes[1].scatter(
        cx, gy, s=MARKER_SIZE, color=GT_COLOR,
        edgecolor="white", linewidth=1, zorder=3
    )

    axes[2].step(
        x_step, full_step, where="post",
        color="#9e9e9e", linewidth=LINE_W, zorder=1
    )
    axes[2].hlines(
        py, sx0, sx1, color=ok_color,
        linewidth=BLOCK_W, zorder=2
    )
    axes[2].scatter(
        cx, py, s=MARKER_SIZE, color=ok_color,
        edgecolor="white", linewidth=1, zorder=3
    )

    for ax in axes[1:]:
        for x in cx:
            ax.axvline(
                x, color=GT_COLOR, linestyle="--",
                linewidth=0.8, alpha=0.4, zorder=0
            )

    axes[0].set_title(f"Real all night | {subject}")
    axes[1].set_title(
        f"Ground Truth (sequences used) | {subject}"
    )
    axes[2].set_title(
        f"Soft Voting (Exp23) Predicted | {subject} | "
        f"Correct {int(correct.sum())}/{len(sub)} "
        f"({correct.mean():.1%})"
    )

    for ax in axes:
        ax.set_ylabel("Sleep stage")
        ax.set_yticks(list(stage_y.values()))
        ax.set_yticklabels(list(stage_y.keys()))
        ax.set_ylim(-0.5, 4.5)
        ax.xaxis.set_major_locator(MultipleLocator(0.5))
        ax.grid(True, alpha=0.3)

    axes[0].set_xlim(-0.05, x_end[-1] * 1.12)

    axes[1].legend(handles=[
        Line2D(
            [0], [0], color="#9e9e9e",
            lw=LINE_W, label="Real all night"
        ),
        Line2D(
            [0], [0], color=GT_COLOR,
            lw=4, label="Ground Truth (selected sequence)"
        ),
    ], loc="lower right", framealpha=0.95)

    axes[2].set_xlabel("Time since recording start (hours)")
    axes[2].legend(handles=[
        Line2D(
            [0], [0], color="#9e9e9e",
            lw=LINE_W, label="Real all night"
        ),
        Line2D(
            [0], [0], color="green",
            lw=4, label="Predicted: Correct"
        ),
        Line2D(
            [0], [0], color="red",
            lw=4, label="Predicted: Incorrect"
        ),
    ], loc="lower right", framealpha=0.95)

    plt.tight_layout()
    plt.savefig(
        hyp_dir / f"hypnogram_full_{subject}.png",
        dpi=150, bbox_inches="tight"
    )
    plt.show()
    plt.close()

## 9.1 สัญญาณดิบทั้ง subject เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1

def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"

stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

plot_subjects = sorted(soft_table["subject"].unique())
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")

signal_dir = RESULT_DIR / "signal_whole_recording"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN

for subject in plot_subjects:
    sub = soft_table[soft_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep]

    xs, los, his = [], [], []
    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        seg = np.asarray(X_all[i][:n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        xs.append(np.append(t0 + (np.arange(n_bin) + 0.5) * BIN / FS / 3600, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))

    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(20, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    for _, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        t1 = (r["epoch_end"] + 1) * 30 / 3600
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for k in range(n_sig):
            axes[k].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", alpha=0.1, linewidth=0, zorder=0)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Soft Voting (Exp23)"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(sub["epoch_start"].min() * 30 / 3600, (sub["epoch_end"].max() + 1) * 30 / 3600)
    axes[-1].set_xlabel("Time since recording start (hours)")

    axes[0].legend(
        handles=[Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES] + [
            Line2D([0], [0], color="green", lw=8, label="Correct"),
            Line2D([0], [0], color="red", lw=8, label="Wrong"),
        ],
        loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8
    )

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | whole recording | Soft Voting (Exp23)  | "
        f"correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%}) (blank = no sequence)"
    )

    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_whole_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 9.2 สัญญาณของแต่ละ sequence เทียบระยะการนอนจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
SEQ_PER_FIG = 20
MODE = "pages"
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1
FIG_W = 24
GAP = 0.35

stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

plot_subjects = sorted(soft_table["subject"].unique())
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")

signal_dir = RESULT_DIR / "signal_sequences"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN
seq_sec = X_all.shape[1] / FS

for subject in plot_subjects:
    sub = soft_table[soft_table["subject"] == subject].sort_values("epoch_start").reset_index(drop=True)

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep].reset_index(drop=True)

    all_rows = list(sub.iterrows())
    n_total = len(all_rows)
    if n_total == 0:
        continue

    n_ok_all = int(sub["correct"].sum())

    if MODE == "even" and n_total > SEQ_PER_FIG:
        pick = np.linspace(0, n_total - 1, SEQ_PER_FIG).round().astype(int)
        pages = [[all_rows[j] for j in pick]]
    else:
        pages = [all_rows[p:p + SEQ_PER_FIG] for p in range(0, n_total, SEQ_PER_FIG)]

    for page_no, rows in enumerate(pages, 1):
        n_seq = len(rows)

        xs, los, his = [], [], []
        for k, (i, r) in enumerate(rows):
            x0 = k * (1 + GAP)
            seg = np.asarray(X_all[i][:n_bin * BIN, :])[:, ch_idx]
            seg = seg.reshape(n_bin, BIN, n_sig)
            rel = (np.arange(n_bin) + 0.5) * BIN / FS / seq_sec
            xs.append(np.append(x0 + rel, np.nan))
            los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
            his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))

        x = np.concatenate(xs)
        lo = np.concatenate(los)
        hi = np.concatenate(his)

        fig, axes = plt.subplots(
            n_sig + 3, 1, figsize=(FIG_W, 1.2 * n_sig + 3.5), sharex=True,
            gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
        )

        for k in range(n_sig):
            axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
            lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
            axes[k].set_ylim(-lim, lim)
            axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
            axes[k].grid(alpha=0.3)

        centers, labels = [], []
        for k, (i, r) in enumerate(rows):
            t0, t1 = k * (1 + GAP), k * (1 + GAP) + 1

            axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
            axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
            axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)

            for kk in range(n_sig):
                axes[kk].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", alpha=0.1, linewidth=0, zorder=0)

            axes[-3].text(
                (t0 + t1) / 2, 0.5, r["actual"], ha="center", va="center", fontsize=10,
                fontweight="bold", transform=axes[-3].get_xaxis_transform()
            )
            axes[-2].text(
                (t0 + t1) / 2, 0.5, r["predicted"], ha="center", va="center", fontsize=10,
                fontweight="bold", transform=axes[-2].get_xaxis_transform()
            )

            centers.append((t0 + t1) / 2)
            labels.append(f"{r['epoch_start'] * 30 / 3600:.2f} h\n(epoch {int(r['epoch_start'])}-{int(r['epoch_end'])})")

        for k in range(n_seq - 1):
            xg = k * (1 + GAP) + 1 + GAP / 2
            for ax in axes:
                ax.axvline(xg, color="gray", linestyle=":", linewidth=1)

        for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Hard Voting"), (axes[-1], "Correct?")]:
            ax.set_yticks([])
            ax.set_ylabel(name, rotation=0, ha="right", va="center")

        n_slot = max(n_seq, SEQ_PER_FIG)
        axes[-1].set_xlim(-GAP / 2, n_slot * (1 + GAP) - GAP / 2)
        axes[-1].set_xticks(centers)
        axes[-1].set_xticklabels(labels, fontsize=8)
        axes[-1].set_xlabel("Selected sequences in time order (gaps removed; tick = real time since recording start)")

        axes[0].legend(
            handles=[Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES] + [
                Line2D([0], [0], color="green", lw=8, label="Correct"),
                Line2D([0], [0], color="red", lw=8, label="Wrong"),
            ],
            loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8
        )

        n_ok_shown = sum(bool(r["correct"]) for _, r in rows)

        if MODE == "pages":
            first = (page_no - 1) * SEQ_PER_FIG + 1
            page_txt = f" | page {page_no}/{len(pages)} (sequences {first}-{first + n_seq - 1} of {n_total})"
        else:
            page_txt = ""

        fig.suptitle(
            f"{subject}{page_txt} | Hard Voting | "
            f"shown {n_ok_shown}/{n_seq} correct | "
            f"whole subject {n_ok_all}/{n_total} correct ({n_ok_all / n_total:.1%})"
        )

        plt.tight_layout()
        suffix = "" if MODE == "even" else f"_p{page_no}"
        plt.savefig(signal_dir / f"signal_sequences_{subject}{suffix}.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()

## 10. ตารางภาพรวมทุก subject: ทายถูก/ทั้งหมด ในแต่ละระยะจริง (Test)

In [ ]:
total = soft_table.groupby(["subject", "actual"]).size().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0)

right = soft_table.groupby(["subject", "actual"])["correct"].sum().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0).astype(int)

rate = right / total.where(total > 0, 1)

order = rate.mean(axis=1).sort_values(ascending=False, kind="stable").index
total, right, rate = total.loc[order], right.loc[order], rate.loc[order]
n = len(rate)

green = np.array([0.78, 0.92, 0.78])
red = np.array([0.96, 0.70, 0.70])
colors = red + (green - red) * rate.to_numpy()[:, :, None]

fig, ax = plt.subplots(figsize=(8, 0.4 * n + 2))
ax.imshow(colors, aspect="auto")

for i in range(n):
    for j in range(len(STAGES)):
        ax.text(j, i, f"{right.iloc[i, j]}/{total.iloc[i, j]}", ha="center", va="center", fontsize=9)

ax.set_xticks(range(len(STAGES)))
ax.set_xticklabels([f"Actual {s}\n{right[s].sum()}/{total[s].sum()}" for s in STAGES])
ax.xaxis.tick_top()

ax.set_yticks(range(n))
ax.set_yticklabels([f"{s} ({right.loc[s].sum()}/{total.loc[s].sum()})" for s in rate.index], fontsize=8)

ax.set_title(
    "Test Soft Voting Ensemble: correct/total sequences per actual stage (green = high, red = low)",
    pad=40
)

plt.tight_layout()
plt.savefig(RESULT_DIR / "proof_table_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. รายละเอียดรายคน: สัญญาณ 30 วินาที, ความน่าจะเป็นของโมเดล (Test)

In [ ]:
tables = {"test": soft_table}

def visualize_subject(split, subject):
    meta = DATA[split]["meta"]
    table = tables[split]

    idx = []
    for label in LABELS:
        rows = meta.index[(meta["subject"] == subject) & (meta["label"] == label)].to_numpy()
        if len(rows) == 0:
            print(f"ข้าม {subject}: ไม่มี Sequence ของระยะ {STAGES[label]}")
            return
        idx.append(rows[0])

    t = np.arange(EPOCH_LEN) / FS

    fig, axes = plt.subplots(
        N_CHANNELS + 1, len(idx),
        figsize=(3.3 * len(idx), 2 * (N_CHANNELS + 1)),
        squeeze=False
    )

    for col, i in enumerate(idx):
        row = table.loc[i]
        signal = DATA[split]["X"][i][-EPOCH_LEN:, :]
        color = "green" if row["correct"] else "red"

        axes[0, col].set_title(
            f"Expert: {row['actual']}\nSoft Voting (Exp23): {row['predicted']}",
            color=color, fontweight="bold"
        )

        for c in range(N_CHANNELS):
            axes[c, col].plot(t, signal[:, c], lw=0.6)
            axes[c, col].set_xlim(0, 30)
            axes[c, col].grid(alpha=0.3)

        axes[-1, col].bar(
            ["Correct", "Wrong"],
            [int(row["correct"]), int(not row["correct"])],
            color=["green", "red"]
        )
        axes[-1, col].set_ylim(0, 1)
        axes[-1, col].set_ylabel("Result")

    for c in range(N_CHANNELS):
        low = min(axes[c, col].get_ylim()[0] for col in range(len(idx)))
        high = max(axes[c, col].get_ylim()[1] for col in range(len(idx)))

        for col in range(len(idx)):
            axes[c, col].set_ylim(low, high)

        axes[c, 0].set_ylabel(SELECTED_CHANNELS[c])

    plt.suptitle(f"{split} | {subject} | Last 30 s of the first sequence per stage", y=1.02)
    plt.tight_layout()

    out_dir = RESULT_DIR / "visualize"
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / f"{split}_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()


subject_acc = soft_table.groupby("subject")["correct"].mean().sort_values()

picks = np.linspace(0, len(subject_acc) - 1, min(5, len(subject_acc)), dtype=int)
selected_subjects = subject_acc.index[picks].tolist()

display(subject_acc.loc[selected_subjects])

for subject in selected_subjects:
    visualize_subject("test", subject)

## 12. แถวสรุปผล

In [ ]:
summary_row = pd.DataFrame([{
    "experiment": f"{CANDIDATE}_test",
    "seed": "42_43_44",
    "n_test_subjects": soft_table["subject"].nunique(),
    "n_test_sequences": len(y_test),
    "test_accuracy": round(ens_overall["accuracy"], 4),
    "test_macro_f1": round(ens_overall["macro_f1"], 4),
    "test_kappa": round(ens_overall["cohen_kappa"], 4),
}])

display(summary_row)
summary_row.to_csv(RESULT_DIR / "test_experiment_row.csv",index=False)